# Deployment Strategy Saga Orchestrator — epic proof

Drives `services/deployment-strategy-orchestrator.ts` (the module the cucumber steps use) against the committed multi-cloud-shop fixture: every service's declared strategy compiles to saga steps and health-gated promotion takes all four strategies to green.

In [1]:
import { strict as assert } from "node:assert";

import {
	createStrategyOrchestrator,
	loadDeployFixture,
	runMultiCloudShopCanaryRegression,
	runReferenceAppGreen,
	selectStrategyForService,
	STRATEGY_KINDS,
} from "./services/deployment-strategy-orchestrator.ts";

const fixture = await loadDeployFixture();

// 1) Every service declares a known strategy that compiles to saga steps.
const services = Object.entries(fixture.referenceApp.services);
for (const [serviceId, serviceConfig] of services) {
	const config = serviceConfig as { strategy: string };
	assert.ok(STRATEGY_KINDS.includes(config.strategy));
	const strategy = selectStrategyForService(config);
	assert.equal(strategy.strategy, config.strategy);
	assert.ok(strategy.steps.length > 0);
	console.log(
		`service=${serviceId} strategy=${config.strategy} sagaSteps=${strategy.steps.length}`,
	);
}

// 2) Health-gated promotion takes every strategy to green.
const green = await runReferenceAppGreen(fixture);
assert.equal(green.app, "multi-cloud-shop");
assert.equal(green.allPromoted, true);
assert.equal(green.deployments.length, 4);
for (const deployment of green.deployments) {
	assert.equal(deployment.status, "promoted");
	assert.match(deployment.gatewayState.activeVersion, /@sha256:green$/);
}
console.log(
	`green rollout app=${green.app} allPromoted=${green.allPromoted} deployments=${green.deployments.length}`,
);

service=web strategy=canary sagaSteps=3
service=jobs-worker strategy=progressive sagaSteps=4
service=session-cache strategy=blue-green sagaSteps=3
service=orders-db-migrator strategy=recreate sagaSteps=3
green rollout app=multi-cloud-shop allPromoted=true deployments=4


The gates bite both ways: a seeded canary SLO regression at 50% traffic auto-rolls back to blue with an event-sourced decision, and provider-unhealthy signals roll back what healthy signals promote.

In [2]:
// 3) A seeded canary SLO regression at 50% traffic auto-rolls back to blue.
const regression = await runMultiCloudShopCanaryRegression(fixture);
assert.equal(regression.service, "web");
assert.equal(regression.strategy, "canary");
assert.equal(regression.rolledBack, true);
assert.equal(regression.seededRegressionAtWeight, 50);
assert.equal(regression.eventSourced.metricDecision, "rollback");
assert.ok(regression.eventSourced.rolloutEvents.includes("rollout.RolledBack"));
assert.equal(regression.gatewayState.activeVersion, "web@sha256:blue");
console.log(
	`canary regression service=${regression.service} atWeight=${regression.seededRegressionAtWeight} decision=${regression.eventSourced.metricDecision} gateway=${regression.gatewayState.activeVersion}`,
);

// 4) Health gates decide promotion: healthy signals promote, provider-unhealthy rolls back.
const orchestrator = createStrategyOrchestrator();
const web = fixture.referenceApp.services.web;
const healthy = await orchestrator.deployService({
	serviceId: "web",
	serviceConfig: web,
	scenario: fixture.scenarios.healthyPromotion,
	runId: "example-health-pass",
});
const unhealthy = await orchestrator.deployService({
	serviceId: "web",
	serviceConfig: web,
	scenario: fixture.scenarios.providerUnhealthy,
	runId: "example-health-fail",
});
assert.equal(healthy.status, "promoted");
assert.equal(unhealthy.status, "rolled-back");
console.log(
	`health gate healthy=${healthy.status} providerUnhealthy=${unhealthy.status}`,
);

console.log(
	"EPIC HELD: strategies compile to sagas, promotion is health-gated, regressions roll back",
);

canary regression service=web atWeight=50 decision=rollback gateway=web@sha256:blue
health gate healthy=promoted providerUnhealthy=rolled-back
EPIC HELD: strategies compile to sagas, promotion is health-gated, regressions roll back


Every assertion above ran green — the same invariants the executable `features/*.feature` scenarios enforce in the repo cucumber lane. `examples/usage.ts` is the gate-executed twin of this notebook.